# Notebook 05 — Jacobians for Robotics (EKF practice)

**Goal:** Learn what Jacobians are, compute them numerically and analytically, and validate them for EKF motion/measurement models.


## Table of Contents

- [1) Jacobian intuition + math](#1-jacobian-intuition--math)
- [2) Numerical Jacobian (finite difference)](#2-numerical-jacobian-finite-difference)
- [3) Test case A — motion model Jacobian](#3-test-case-a--motion-model-jacobian)
- [4) Test case B — range/bearing measurement Jacobian](#4-test-case-b--rangebearing-measurement-jacobian)
- [5) Exercise — derive and verify range-bearing Jacobian](#5-exercise--derive-and-verify-range-bearing-jacobian)
- [6) EKF connection (must-remember)](#6-ekf-connection-must-remember)

---

## Navigation

- Previous: [ntbk-04-linearization-and-taylor-approx.ipynb](./ntbk-04-linearization-and-taylor-approx.ipynb)
- Next: [ntbk-06-ekf-core-ideas-predict-update.ipynb](./ntbk-06-ekf-core-ideas-predict-update.ipynb)

## Relevant source files (repo paths)
- `src/kiss_slam/models/motion.py`
- `src/kiss_slam/models/measurement.py`


In [ ]:
# Notebook setup (reproducible + matplotlib defaults)
import numpy as np
import matplotlib.pyplot as plt
from _notebook_utils import set_seed

SEED = 104
set_seed(SEED)
plt.rcdefaults()


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(7)


## 1) Jacobian intuition + math

For a vector function \(f: \mathbb{R}^n \to \mathbb{R}^m\), the Jacobian is:

\[
J(x) = \frac{\partial f}{\partial x} =
\begin{bmatrix}
\frac{\partial f_1}{\partial x_1} & \cdots & \frac{\partial f_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial f_m}{\partial x_1} & \cdots & \frac{\partial f_m}{\partial x_n}
\end{bmatrix}
\]

EKF usage:
- **F** = Jacobian of motion model wrt state.
- **H** = Jacobian of measurement model wrt state.

These matrices propagate covariance and compute Kalman gain.


## 2) Numerical Jacobian (finite difference)

We'll implement `jacobian_fd(func, x)` using central differences:

\[
\frac{\partial f}{\partial x_i} \approx \frac{f(x + \epsilon e_i) - f(x - \epsilon e_i)}{2\epsilon}
\]


In [ ]:
def jacobian_fd(func, x, eps=1e-6):
    """Numerical Jacobian using central finite differences."""
    x = np.asarray(x, dtype=float)
    y0 = np.asarray(func(x), dtype=float)
    m = y0.size
    n = x.size
    J = np.zeros((m, n), dtype=float)

    for i in range(n):
        dx = np.zeros_like(x)
        dx[i] = eps
        y_plus = np.asarray(func(x + dx), dtype=float)
        y_minus = np.asarray(func(x - dx), dtype=float)
        J[:, i] = (y_plus - y_minus) / (2.0 * eps)

    return J


## 3) Test case A — motion model Jacobian

Simple state \(x=[p_x,p_y,\theta]^T\), control \(u=[v,\omega]\), time-step `dt`:

\[
p_x' = p_x + v\,dt\cos\theta,\quad
p_y' = p_y + v\,dt\sin\theta,\quad
\theta' = \theta + \omega dt
\]

Analytic Jacobian wrt state:

\[
F =
\begin{bmatrix}
1 & 0 & -v dt\sin\theta \\
0 & 1 &  v dt\cos\theta \\
0 & 0 & 1
\end{bmatrix}
\]


In [ ]:
def motion_func(x, u, dt):
    px, py, th = x
    v, w = u
    return np.array([
        px + v * dt * np.cos(th),
        py + v * dt * np.sin(th),
        th + w * dt,
    ])


def motion_jacobian_analytic(x, u, dt):
    _, _, th = x
    v, _ = u
    return np.array([
        [1.0, 0.0, -v * dt * np.sin(th)],
        [0.0, 1.0,  v * dt * np.cos(th)],
        [0.0, 0.0, 1.0],
    ])

x = np.array([1.0, 2.0, 0.3])
u = np.array([1.5, 0.2])
dt = 0.1

J_num = jacobian_fd(lambda x_: motion_func(x_, u, dt), x)
J_an = motion_jacobian_analytic(x, u, dt)

print("Motion Jacobian (numeric):\n", J_num)
print("Motion Jacobian (analytic):\n", J_an)
print("max abs diff:", np.max(np.abs(J_num - J_an)))


## 4) Test case B — range/bearing measurement Jacobian

For robot pose \((p_x,p_y,\theta)\) and landmark \((l_x,l_y)\):

\[
\Delta x = l_x - p_x,\quad \Delta y = l_y - p_y,\quad q = \Delta x^2 + \Delta y^2
\]
\[
r = \sqrt{q},\quad \phi = \arctan2(\Delta y,\Delta x) - \theta
\]

Measurement \(z=[r,\phi]^T\). Jacobian wrt robot state \([p_x,p_y,\theta]\):

\[
H = \begin{bmatrix}
-\Delta x/\sqrt{q} & -\Delta y/\sqrt{q} & 0 \\
\Delta y/q & -\Delta x/q & -1
\end{bmatrix}
\]


In [ ]:
def wrap_angle(a):
    return (a + np.pi) % (2.0 * np.pi) - np.pi


def measurement_func_robot_state(x_robot, landmark):
    px, py, th = x_robot
    lx, ly = landmark
    dx = lx - px
    dy = ly - py
    r = np.sqrt(dx**2 + dy**2)
    b = wrap_angle(np.arctan2(dy, dx) - th)
    return np.array([r, b])


def measurement_jacobian_analytic(x_robot, landmark):
    px, py, th = x_robot
    lx, ly = landmark
    dx = lx - px
    dy = ly - py
    q = dx**2 + dy**2
    r = np.sqrt(q)

    return np.array([
        [-dx / r, -dy / r, 0.0],
        [ dy / q, -dx / q, -1.0],
    ])

x_robot = np.array([2.0, 1.0, 0.4])
landmark = np.array([5.0, 4.0])

J_num = jacobian_fd(lambda xr: measurement_func_robot_state(xr, landmark), x_robot)
J_an = measurement_jacobian_analytic(x_robot, landmark)

print("Measurement Jacobian (numeric):\n", J_num)
print("Measurement Jacobian (analytic):\n", J_an)
print("max abs diff:", np.max(np.abs(J_num - J_an)))


## 5) Exercise — derive and verify range-bearing Jacobian

1. Derive \(H\) by hand from the equations above.
2. Implement your own analytic function.
3. Compare against `jacobian_fd` at random robot/landmark pairs.

<details>
<summary>Optional solution outline</summary>

- Use chain rule with \(q=\Delta x^2+\Delta y^2\).
- Check edge cases where `q` is very small (robot near landmark).
- Batch-test with random points and report max error.

```python
for _ in range(100):
    xr = np.array([np.random.uniform(-5,5), np.random.uniform(-5,5), np.random.uniform(-np.pi,np.pi)])
    lm = np.array([np.random.uniform(-5,5), np.random.uniform(-5,5)])
    if np.linalg.norm(lm - xr[:2]) < 0.3:
        continue
    Jn = jacobian_fd(lambda x: measurement_func_robot_state(x, lm), xr)
    Ja = measurement_jacobian_analytic(xr, lm)
    assert np.max(np.abs(Jn - Ja)) < 1e-4
```
</details>


## 6) EKF connection (must-remember)

In EKF-SLAM:
- `F` linearizes motion to propagate uncertainty in **predict**.
- `H` linearizes measurement to compute innovation covariance and Kalman gain in **update**.

If analytic Jacobians are wrong, EKF can look "okay" at first and then diverge.
A fast finite-difference checker is a practical debugging superpower.


## Exercise Solutions

<details>
<summary>Show solution sketches</summary>

- Re-run the exercise code cells and compare your intermediate values to the reference outputs printed in this notebook.
- Check shapes (`mean`, `covariance`, Jacobians) first; most EKF mistakes are shape/sign issues.
- Use the same `SEED` from the setup cell to keep your run deterministic while debugging.

</details>
